## Model Selection

Before we move on we would like to choose the best possible model for each of the two cases: daily forecast and hourly forecast. It seems that for the daily forecast in the long term you choose linear_order2 in the short to medium term potentially hybrid_order2. For the hourly forecast it seems that just using XGBoost is the best possible model.

The main question is that in the daily forecast if we were to optimise the hyperparameters of the XGBoost model within the hybrid model would it make hybrid_order2 better than linear_order2 in both the short and long term. It would also be nice to do hyperparamter optimisation for the hourly forecast as well just to see whether we can improve the predictions or not. 

Finally it would be good to look at SHAP values to see if we can drop any of the features, particualarly some of the lags as they can make the models computationally expensive.

For our own use (maybe delete later): http://kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

https://hyperopt.github.io/hyperopt/?source=post_page

https://github.com/hyperopt/hyperopt/wiki/FMin

In [24]:
from hyperopt import hp, fmin, tpe, hp, STATUS_OK, Trials
import xgboost as xgb
import pickle
import pandas as pd

In [25]:
# First reload the significant lags
with open("sig_lags_daily.pkl", "rb") as f:
    daily_lags = pickle.load(f)

with open("sig_lags_hourly.pkl", "rb") as f:
    hourly_lags = pickle.load(f)

In [26]:
# Get both the full daily and hourly time series
dir_path = "../data/processed/"
df_daily = pd.read_csv(f"{dir_path}ts_daily2011-2025.csv")
df_hourly = pd.read_csv(f"{dir_path}ts_hour2011-2025.csv")

# Convert dates to datetime objects
df_daily["pickup_date"] = pd.to_datetime(df_daily["pickup_date"])
df_hourly["dt"] = pd.to_datetime(df_hourly["dt"])


In [27]:
# To pass the time series through our helper functions they need to be a pandas series indexed by a datetime object:
ts_hourly = df_hourly["trips"]
ts_hourly.index = df_hourly["dt"]

ts_daily = df_daily["trips"]
ts_daily.index = df_daily["pickup_date"]

In [28]:
# We now need to split into test and train data, we will train on the pre 2024 data and test on 2024 onwards, approx a 90:10 split
ts_daily_train = ts_daily[:"2023-12-31"]
ts_daily_test = ts_daily["2024-01-01":]

ts_hourly_train = ts_hourly[:"2023-12-31"]
ts_hourly_test = ts_hourly["2024-01-01":]

In [20]:
# Define search space
space = {'max_depth': hp.quniform('max_depth', 3, 18, 1),
          'gamma': hp.uniform ('gamma', 1, 9),
          'reg_alpha': hp.quniform('reg_alpha', 40, 180, 1),
          'reg_lambda': hp.uniform('reg_lambda', 0, 1),
          'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
          'min_child_weight': hp.quniform('min_child_weight', 0, 10, 1),
          'n_estimators': 180,
          'seed': 37}

In [21]:
# Define objective function

def objective(space):
    clf = xgb.XGBClassifier(
        n_estimators = space['n_estimators'],
        max_depth = int(space['max_depth']),
        gamma = space['gamma'],
        reg_alpha = space['reg_alpha'],
        reg_lambda = space['reg_lambda'],
        colsample_bytree = int(space['colsample_bytree']),
        min_child_weight = int(space['min_child_weight']))

    evaluation = [(ts_daily_train, y_train), (X_test, y_test)]

    clf.fit(X_train, y_train,
            eval_set = evaluation, eval_metric = "auc",
            early_stopping_rounds = 10, verbose = False)

    pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, pred > 0.5)
    print("SCORE:", accuracy)
    return {'loss': -accuracy, 'status': STATUS_OK}

In [22]:
# Optimisation algorithm
trials = Trials()

best_hyperparams = fmin(fn = objective,
                        space = space,
                        algo = tpe.suggest,
                        max_evals = 100,
                        trials = trials)

  0%|                             | 0/100 [00:00<?, ?trial/s, best loss=?]

job exception: name 'X_train' is not defined



  0%|                             | 0/100 [00:00<?, ?trial/s, best loss=?]


NameError: name 'X_train' is not defined

In [ ]:
print("The best hyperparamters are: ", "\n")
print(best_hyperparams)